# Phase 3: Heuristic-Based Recommendation (Baseline)

Now that we have a **Bipartite Graph**, we can start predicting links. 

In this phase, we use **Heuristics**—simple mathematical rules based on the graph's structure. The most common rule for a recommender is: 
> "If User A and User B both liked Movie X and Movie Y, they probably have similar tastes. Therefore, User A might like Movie Z which User B liked."

## Step 1: User-User Similarity (The Jaccard Coefficient)

How do we measure if two users are 'similar' in a graph?
We look at their **Neighborhoods** (the set of movies they've liked).

The **Jaccard Coefficient** is a standard metric for this:
$$J(U_1, U_2) = \frac{|N(U_1) \cap N(U_2)|}{|N(U_1) \cup N(U_2)|}$$

- **Intersection $(\cap)$**: Movies both users liked.
- **Union $(\cup)$**: Total unique movies liked by both users combined.

A score of 1.0 means they liked exactly the same movies. A score of 0.0 means they have no common interests.

In [1]:
import pandas as pd
import networkx as nx
from collections import Counter

# 1. Re-construct the graph with metadata
df = pd.read_csv('data/u.data', sep='\t', names=['u_id', 'm_id', 'rating', 'ts'])
pos_df = df[df['rating'] >= 4].copy()
pos_df['u'] = 'u_' + pos_df['u_id'].astype(str)
pos_df['m'] = 'm_' + pos_df['m_id'].astype(str)

G = nx.Graph()
G.add_edges_from(zip(pos_df['u'], pos_df['m']))

# Load movie titles for readability
items = pd.read_csv('data/u.item', sep='|', names=['m_id', 'title', 'r_date', 'v_date', 'url'] + [f'g_{i}' for i in range(19)], encoding='latin-1')
titles = { f"m_{row['m_id']}": row['title'] for _, row in items.iterrows() }
nx.set_node_attributes(G, titles, 'title')

def get_jaccard_similarity(user1, user2, graph):
    """Calculates Jaccard similarity between two user nodes."""
    try:
        movies1 = set(graph.neighbors(user1))
        movies2 = set(graph.neighbors(user2))
    except:
        return 0.0
    
    intersection = len(movies1.intersection(movies2))
    union = len(movies1.union(movies2))
    
    return intersection / union if union > 0 else 0.0

print("Graph re-initialized with movie titles.")

Graph re-initialized with movie titles.


## Step 2: Identifying the 'Peer Group'

To make a recommendation for a specific user, we first find their **Peer Group**—the set of other users who have the most similar interests.

In this step, we will:
1. Pick a target user (`u_1`).
2. Compare them against every other user in the graph.
3. Rank all other users by their similarity score.
4. Select the **Top 5** most similar peers.

In [2]:
def find_top_peers(target_user, graph, top_k=5):
    """Finds the users most similar to the target user based on Jaccard similarity."""
    all_users = [n for n in graph.nodes if n.startswith('u_')]
    
    similarities = []
    for other_user in all_users:
        if other_user == target_user: 
            continue
        
        sim = get_jaccard_similarity(target_user, other_user, graph)
        similarities.append((other_user, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]

target = 'u_1'
top_peers = find_top_peers(target, G, top_k=5)
print(f"Top Peers for {target}: {top_peers}")

Top Peers for u_1: [('u_457', 0.30689655172413793), ('u_889', 0.29365079365079366), ('u_864', 0.2892857142857143), ('u_429', 0.28308823529411764), ('u_738', 0.27835051546391754)]


## Step 3: Generating Recommendations (Link Candidates)

Now that we know who our target user's 'soulmates' are, we can see what movies they liked that our target user hasn't seen yet.

This is the **Link Prediction** moment: we are predicting that an edge *should* exist between `u_1` and these new movies.

**The Logic:**
1.  **Peer Likes:** Aggregate all movies liked by the top 5 peers.
2.  **Filter Existing Edges:** Remove movies the target user has already rated (we don't recommend what you've already seen).
3.  **Frequency Score:** If 4 out of 5 peers liked a movie, it's a stronger recommendation than if only 1 peer liked it.

In [5]:
def get_recommendations(target_user, top_peers, graph, top_n=5):
    """Generates movie recommendations based on peer likes."""
    # 1. Movies liked by the target user (already seen)
    seen_movies = set(graph.neighbors(target_user))
    
    # 2. Collect all movies liked by peers
    candidate_movies = []
    for peer, sim_score in top_peers:
        peer_movies = set(graph.neighbors(peer))
        # Only consider movies the target user hasn't seen
        new_discoveries = peer_movies - seen_movies
        candidate_movies.extend(list(new_discoveries))
    
    # 3. Rank by frequency (how many peers liked each movie)
    movie_counts = Counter(candidate_movies)
    top_recommendations = movie_counts.most_common(top_n)
    
    return top_recommendations

recommendations = get_recommendations(target, top_peers, G, top_n=5)

print(f"\n--- Top 5 Recommendations for {target} ---")
for movie_node, freq in recommendations:
    title = G.nodes[movie_node].get('title', 'Unknown Title')
    print(f"- {title} (Recommended by {freq} similar users)")


--- Top 5 Recommendations for u_1 ---
- Tombstone (1993) (Recommended by 5 similar users)
- Glory (1989) (Recommended by 5 similar users)
- Abyss, The (1989) (Recommended by 5 similar users)
- One Flew Over the Cuckoo's Nest (1975) (Recommended by 5 similar users)
- Schindler's List (1993) (Recommended by 5 similar users)
